# Stage 2D — Bi-LSTM Training
### ISL Real-Time Sign Language Translator

**What this notebook does:**
1. Mount Google Drive and load extracted landmarks
2. Inspect and visualize the dataset
3. Build a Bi-LSTM sequence classifier
4. Train with early stopping
5. Plot loss and accuracy curves
6. Evaluate on the test set
7. Save the trained model

**Expected input files in Google Drive:**
- `landmarks_index.csv`
- `label_map.csv`
- `extracted_landmarks.zip` (or `extracted_landmarks/` folder)

**Runtime:** Set to T4 GPU → Runtime → Change runtime type → T4 GPU

In [13]:
from google.colab import drive
drive.mount('/content/drive')
import os

drive_dir = "/content/drive/MyDrive"
landmarks_zip = os.path.join(drive_dir, 'extracted_landmarks.zip')
index_csv = os.path.join(drive_dir, 'landmarks_index.csv')
label_map_csv = os.path.join(drive_dir, 'label_map.csv')
extract_dir = '/content/extracted_landmarks'
model_save_dir = '/content/drive/MyDrive'

os.makedirs(model_save_dir, exist_ok=True)

if not os.path.exists(extract_dir):
  print("Unzipping extracted_landmarks.zip...")
  os.system(f"unzip -q {landmarks_zip} -d/content")
  print("Done")
else:
  print("extracted_landmarks/ already exists, skipping unzip.")
print("Drive mounted and paths configured.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
extracted_landmarks/ already exists, skipping unzip.
Drive mounted and paths configured.


## Cell 2 — Imports and GPU Check

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
  print(f"GPU: {torch.cuda.get_device_name(0)}")
  print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
GPU: Tesla T4
Memory: 15.6 GB


## Cell 3 — Load and Inspect Dataset

In [18]:
index_df = pd.read_csv(index_csv)
label_map = pd.read_csv(label_map_csv)

index_df['npy_path'] = index_df['video_path'].apply(
    lambda p: str(Path(extract_dir) / Path(p).with_suffix('.npy'))
)
print(f'Total sequences: {len(index_df)}')
print(f"Classses: {index_df['class_index'].nunique()}")
print()
print("split breakdown: ")
print(index_df['split'].value_counts())
print()

lengths = []
#for path in index_df['npy_path']:
  #try:
    #seq = np.load(path)

Total sequences: 943
Classses: 50

split breakdown: 
split
train    675
test     191
val       77
Name: count, dtype: int64



In [19]:
index_df.head()

,video_path,npy_path,label,split,class_index
0,Adjectives/1. loud/MVI_9448.MOV,/content/extracted_landmarks/Adjectives/1. lou...,1. loud,train,1.0
1,Places/19. House/MVI_3515.MOV,/content/extracted_landmarks/Places/19. House/...,19. House,train,5.0
2,Adjectives/83. big large/MVI_5204.MOV,/content/extracted_landmarks/Adjectives/83. bi...,83. big large,train,41.0
3,Jobs/91. Priest/MVI_5345.MOV,/content/extracted_landmarks/Jobs/91. Priest/M...,91. Priest,train,46.0
4,Places/23. Court/MVI_3612.MOV,/content/extracted_landmarks/Places/23. Court/...,23. Court,train,8.0


In [20]:
label_map.head()

,label,class_index
0,1. Dog,0
1,1. loud,1
2,11. Car,2
3,14. Election,3
4,16. train ticket,4
